# Datasets Lake — one notebook, four formats, the whole Spotify silver table

B1.8 data-science layer. Reads the **music** dataset's silver data in all four lakeFS formats — **Parquet · Arrow · Avro · Lance** — straight from the in-cluster lakeFS S3 gateway (no port-forward), then does real analysis with **polars** and **DuckDB SQL**. `LAKEFS_*` env is injected by JupyterHub.

In [ ]:
import io, os, time
import polars as pl
import s3fs

LAKEFS_ENDPOINT = os.environ.get('LAKEFS_ENDPOINT', 'http://lakefs.data-mesh.svc.cluster.local:8000')
KEY, SECRET = os.environ['LAKEFS_ACCESS_KEY_ID'], os.environ['LAKEFS_SECRET_ACCESS_KEY']
REPO, BRANCH, TABLE = 'music', 'main', 'spotify_tracks'

# object_store options (Rust readers: parquet, lance) — path-style, plain http
SO = {'access_key_id': KEY, 'secret_access_key': SECRET, 'endpoint': LAKEFS_ENDPOINT,
      'allow_http': 'true', 'region': 'us-east-1'}
# s3fs (byte-stream readers: arrow, avro) — used to list + pull objects through the gateway
_fs = s3fs.S3FileSystem(key=KEY, secret=SECRET, use_ssl=False,
                        client_kwargs={'endpoint_url': LAKEFS_ENDPOINT})

def _first(fmt, ext):
    return next(f for f in _fs.ls(f'{REPO}/{BRANCH}/{fmt}/{TABLE}') if f.endswith(ext))

def read_parquet():
    return pl.read_parquet(f"s3://{_first('parquet', '.parquet')}", storage_options=SO)
def read_arrow():
    # read the IPC bytes through s3fs then parse locally (polars' S3 IPC path wants different creds keys)
    with _fs.open(_first('arrow', '.arrow'), 'rb') as f:
        return pl.read_ipc(io.BytesIO(f.read()))
def read_avro():
    with _fs.open(_first('avro', '.avro'), 'rb') as f:
        return pl.read_avro(io.BytesIO(f.read()))
def read_lance():
    import lance
    return pl.from_arrow(lance.dataset(f's3://{REPO}/{BRANCH}/lance/{TABLE}', storage_options=SO).to_table())

print('lakeFS:', LAKEFS_ENDPOINT)

## 1 · All four formats, same data — with read timings

In [ ]:
frames, rows = {}, []
for name, fn in [('parquet', read_parquet), ('arrow', read_arrow), ('avro', read_avro), ('lance', read_lance)]:
    t = time.perf_counter()
    try:
        df = fn(); dt = time.perf_counter() - t
        frames[name] = df
        rows.append({'format': name, 'rows': df.height, 'cols': df.width, 'read_seconds': round(dt, 2)})
    except Exception as e:
        rows.append({'format': name, 'rows': None, 'cols': None, 'read_seconds': None, 'error': f'{type(e).__name__}: {e}'})
pl.DataFrame(rows)

## 2 · Schema & summary statistics

In [ ]:
df = frames['parquet']
print('columns:', df.columns)
df.head(8)

In [ ]:
df.describe()

## 3 · Genre landscape — top 20 by track count, with mean audio profile

In [ ]:
feats = [c for c in ('danceability','energy','valence','acousticness','tempo','popularity') if c in df.columns]
gcol = 'track_genre' if 'track_genre' in df.columns else df.columns[-1]
(df.group_by(gcol)
   .agg(pl.len().alias('tracks'), *[pl.col(f).mean().round(3).alias(f) for f in feats])
   .sort('tracks', descending=True)
   .head(20))

## 4 · Audio-feature correlations + the extremes

In [ ]:
acol = [c for c in ('danceability','energy','valence','acousticness','instrumentalness','liveness','speechiness','loudness','tempo') if c in df.columns]
df.select(acol).corr().with_columns(feature=pl.Series(acol)).select(['feature', *acol])

In [ ]:
show = [c for c in ('track_name','artists',gcol,'danceability','energy','valence','popularity') if c in df.columns]
for metric in ('danceability','energy','valence','popularity'):
    if metric in df.columns:
        print(f'\n--- highest {metric} ---')
        print(df.select(show).sort(metric, descending=True).head(5))

## 5 · DuckDB SQL over the same lakeFS Parquet — incl. a window function

In [ ]:
import duckdb
con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute(f"SET s3_endpoint='{LAKEFS_ENDPOINT.replace('http://','')}'; SET s3_use_ssl=false; SET s3_url_style='path'; SET s3_region='us-east-1'; SET s3_access_key_id='{KEY}'; SET s3_secret_access_key='{SECRET}';")
PQ = f"s3://{_first('parquet', '.parquet')}"
con.execute(f"SELECT {gcol} AS genre, count(*) tracks, round(avg(danceability),3) danceability, round(avg(energy),3) energy FROM read_parquet('{PQ}') GROUP BY 1 ORDER BY tracks DESC LIMIT 10").pl()

In [ ]:
# the single most popular track in each of the 12 biggest genres (window function)
con.execute(f"""
WITH ranked AS (
  SELECT {gcol} genre, track_name, artists, popularity,
         row_number() OVER (PARTITION BY {gcol} ORDER BY popularity DESC) rn,
         count(*)    OVER (PARTITION BY {gcol}) genre_size
  FROM read_parquet('{PQ}')
) SELECT genre, track_name, artists, popularity FROM ranked WHERE rn = 1 ORDER BY genre_size DESC LIMIT 12
""").pl()

## 6 · Visuals

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, feat in zip(ax, ['danceability', 'energy', 'valence']):
    a.hist(df[feat].to_numpy(), bins=40, color='#4C78A8'); a.set_title(f'{feat} distribution')
plt.tight_layout(); plt.show()

In [ ]:
top = df.group_by(gcol).len().sort('len', descending=True).head(12)
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top[gcol].to_list()[::-1], top['len'].to_list()[::-1], color='#F58518')
ax.set_title('Top 12 genres by track count'); plt.tight_layout(); plt.show()